In [1]:
# ============================================================================
# notebook: notebooks/00_gatekeeping.ipynb  (clean rebuild, new paths)
# Project: "Incidental vs. Engineered Approval" — cross-group audit of approval quality
# Stage 0: data load + VIP label + three viability gates.
#   A. Circularity: label-axis (PAY_*) vs audit-axis leakage
#   B. Sample: dual-VIP + disadvantaged intersectional cells testable?
#   C. Axis independence (toy) — deferred to Stage 2 for real SHAP
# Writes results/. Run from notebooks/.
# ============================================================================


# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Paths, imports, config
# ─────────────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold

ROOT    = Path("..").resolve()
DATA    = ROOT / "data"
RESULTS = ROOT / "results"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

LEAK_R_THRESHOLD = 0.50
MIN_CELL_N       = 30
BORDERLINE_BAND  = (0.40, 0.60)

LABEL_AXIS = ["PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]
AUDIT_AXIS = ["LIMIT_BAL"] + [f"BILL_AMT{i}" for i in range(1,7)] + [f"PAY_AMT{i}" for i in range(1,7)]
CROSS_AXIS = ["SEX","AGE","MARRIAGE","EDUCATION"]


# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Load Taiwan data (from data/taiwan_raw.parquet, prepared once)
# If the raw parquet is absent, fetch from UCI and standardize column names.
# ─────────────────────────────────────────────────────────────────────────
raw_path = DATA / "taiwan_raw.parquet"
if raw_path.exists():
    df = pd.read_parquet(raw_path)
else:
    from ucimlrepo import fetch_ucirepo
    d = fetch_ucirepo(id=350)
    df = pd.concat([d.data.features, d.data.targets], axis=1)
    xmap = {"X1":"LIMIT_BAL","X2":"SEX","X3":"EDUCATION","X4":"MARRIAGE","X5":"AGE",
            "X6":"PAY_0","X7":"PAY_2","X8":"PAY_3","X9":"PAY_4","X10":"PAY_5","X11":"PAY_6",
            "X12":"BILL_AMT1","X13":"BILL_AMT2","X14":"BILL_AMT3","X15":"BILL_AMT4",
            "X16":"BILL_AMT5","X17":"BILL_AMT6","X18":"PAY_AMT1","X19":"PAY_AMT2",
            "X20":"PAY_AMT3","X21":"PAY_AMT4","X22":"PAY_AMT5","X23":"PAY_AMT6"}
    if "X1" in df.columns: df = df.rename(columns=xmap)
    tgt = [c for c in df.columns if c.lower().startswith(("y","default"))][-1]
    df = df.rename(columns={tgt:"DEFAULT"})
    df.to_parquet(raw_path)

print("Shape:", df.shape, "| missing:", int(df.isna().sum().sum()))
for name, cols in [("LABEL",LABEL_AXIS),("AUDIT",AUDIT_AXIS),("CROSS",CROSS_AXIS)]:
    assert all(c in df.columns for c in cols), f"{name} axis missing columns"
assert set(LABEL_AXIS).isdisjoint(AUDIT_AXIS), "Label/audit overlap!"
print("Axis separation OK.")


# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — VIP label: never delinquent AND top-half repayment diligence
# ─────────────────────────────────────────────────────────────────────────
never_delinquent = (df[LABEL_AXIS] <= 0).all(axis=1)
diligence = pd.concat([df[f"PAY_AMT{t}"]/(df[f"BILL_AMT{t}"].abs()+1.0) for t in range(1,7)],
                      axis=1).mean(axis=1)
df["VIP_CLEAR"] = (never_delinquent & (diligence >= diligence.median())).astype(int)
print(f"VIP_CLEAR rate = {df['VIP_CLEAR'].mean():.4f}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — VERDICT A: circularity / label leakage
# ─────────────────────────────────────────────────────────────────────────
leak = {v: max(abs(np.corrcoef(df[v], df[c])[0,1]) for c in LABEL_AXIS) for v in AUDIT_AXIS}
leak_s = pd.Series(leak).sort_values(ascending=False)
AUDIT_CLEAN = [v for v in AUDIT_AXIS if leak_s[v] <= LEAK_R_THRESHOLD]
print("Max |corr| with label axis (top 3):", leak_s.head(3).round(3).to_dict())
print(f"Audit vars surviving (|r|<={LEAK_R_THRESHOLD}): {len(AUDIT_CLEAN)}/{len(AUDIT_AXIS)}")
print(">>> VERDICT A:", "PASS" if len(AUDIT_CLEAN)>=6 else "REVIEW")


# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — Borderline group via audit-only RF out-of-fold probabilities
# ─────────────────────────────────────────────────────────────────────────
X = df[AUDIT_CLEAN].values
y = df["VIP_CLEAR"].values
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                            random_state=RANDOM_STATE, n_jobs=-1)
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
df["P_VIP"] = cross_val_predict(rf, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:,1]
lo, hi = BORDERLINE_BAND
df["VIP_BORDERLINE"] = ((df["P_VIP"]>=lo) & (df["P_VIP"]<=hi)).astype(int)
df["APPROVED"] = df["VIP_CLEAR"]
print(f"Borderline cases: {int(df['VIP_BORDERLINE'].sum())}, "
      f"approved&borderline: {int(((df.VIP_BORDERLINE==1)&(df.APPROVED==1)).sum())}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — VERDICT B: disadvantaged intersectional cells (SEX x AGE x EDU)
# ─────────────────────────────────────────────────────────────────────────
df["AGE_BAND"] = pd.cut(df["AGE"], bins=[20,30,40,50,60,100],
                        labels=["20s","30s","40s","50s","60+"], right=False)
df["SEX_LBL"] = df["SEX"].map({1:"M",2:"F"})
edu_map = {1:"grad",2:"univ",3:"hs",4:"other",5:"other",6:"other",0:"other"}
df["EDU_BAND"] = df["EDUCATION"].map(edu_map).fillna("other")

cs = (df.groupby(["SEX_LBL","AGE_BAND","EDU_BAND"], observed=True)
        .agg(n=("APPROVED","size"), approvals=("APPROVED","sum"),
             approval_rate=("APPROVED","mean"), mean_limit=("LIMIT_BAL","mean"),
             borderline_appr=("VIP_BORDERLINE",
                 lambda s: int(((s==1)&(df.loc[s.index,"APPROVED"]==1)).sum())))
        .reset_index())
lo_rate  = cs["approval_rate"] <= cs["approval_rate"].median()
lo_limit = cs["mean_limit"]    <= cs["mean_limit"].median()
cs["disadvantaged"] = lo_rate & lo_limit
dis = cs[cs["disadvantaged"]]
testable = dis[dis["borderline_appr"] >= MIN_CELL_N]
print(f"Disadvantaged cells: {len(dis)}, testable on borderline (>={MIN_CELL_N}): {len(testable)}")
print(">>> VERDICT B:", "PASS" if len(testable)>=1 else "REVIEW")


# ─────────────────────────────────────────────────────────────────────────
# CELL 7 — Persist Stage-0 artifacts
# ─────────────────────────────────────────────────────────────────────────
df.to_parquet(RESULTS / "stage0_labeled.parquet")
cs.to_csv(RESULTS / "stage0_cell_stats.csv", index=False)
print("Saved Stage-0 artifacts to results/.")
print("=== STAGE 0 COMPLETE ===  A:PASS(leakage low)  B:PASS(cells testable)")

Shape: (30000, 27) | missing: 0
Axis separation OK.
VIP_CLEAR rate = 0.3696
Max |corr| with label axis (top 3): {'LIMIT_BAL': 0.296, 'BILL_AMT5': 0.291, 'BILL_AMT6': 0.285}
Audit vars surviving (|r|<=0.5): 13/13
>>> VERDICT A: PASS
Borderline cases: 2086, approved&borderline: 1144
Disadvantaged cells: 14, testable on borderline (>=30): 6
>>> VERDICT B: PASS
Saved Stage-0 artifacts to results/.
=== STAGE 0 COMPLETE ===  A:PASS(leakage low)  B:PASS(cells testable)
